In [13]:
import random
import math
import pandas as pd

In [14]:
def simular_tiempos_mm1(lambda_tasa, mu_tasa, num_clientes):
    """
    1.- Algoritmo para simular los tiempos de llegada, tiempos de servicio
    y tiempos de fin de servicio de una línea de espera M/M/1.
    """
    tiempos_llegada = []
    tiempos_servicio = []
    tiempos_fin_servicio = []

    # 1. Generación de tiempos de llegada (Proceso de Poisson)
    tiempo_actual_llegada = 0.0
    for i in range(num_clientes):
        u_llegada = random.random()
        tiempo_entre_llegada = -math.log(u_llegada) / lambda_tasa
        tiempo_actual_llegada += tiempo_entre_llegada
        tiempos_llegada.append(tiempo_actual_llegada)

    # 2. Generación de duraciones de servicio (Exponencial mu)
    for i in range(num_clientes):
        v_servicio = random.random()
        duracion_servicio = -math.log(v_servicio) / mu_tasa
        tiempos_servicio.append(duracion_servicio)

    # 3. Cálculo de los tiempos de fin de servicio en base a la ocupación del servidor
    tiempo_fin_anterior = 0.0
    for i in range(num_clientes):
        tiempo_llegada_cliente = tiempos_llegada[i]
        duracion_servicio_cliente = tiempos_servicio[i]

        if tiempo_llegada_cliente > tiempo_fin_anterior:
            tiempo_inicio_servicio = tiempo_llegada_cliente
        else:
            tiempo_inicio_servicio = tiempo_fin_anterior

        tiempo_fin_cliente = tiempo_inicio_servicio + duracion_servicio_cliente
        tiempos_fin_servicio.append(tiempo_fin_cliente)
        tiempo_fin_anterior = tiempo_fin_cliente

    return tiempos_llegada, tiempos_servicio, tiempos_fin_servicio

In [15]:
def calcular_desglose_en_tiempo_t(t, tiempos_llegada, tiempos_servicio, tiempos_fin_servicio):
    """
    NUEVO ALGORITMO: Calcula el número de clientes en el sistema (atendidos + cola)
    y el número de clientes esperando estrictamente en la cola en el tiempo t.
    """
    clientes_en_sistema = 0
    clientes_en_cola = 0

    num_clientes = len(tiempos_llegada)
    tiempo_fin_anterior = 0.0

    for i in range(num_clientes):
        t_llegada = tiempos_llegada[i]
        t_fin = tiempos_fin_servicio[i]

        # Reconstruimos el tiempo exacto en que inició el servicio del cliente i
        if t_llegada > tiempo_fin_anterior:
            t_inicio = t_llegada
        else:
            t_inicio = tiempo_fin_anterior

        # Condición 1: ¿El cliente está actualmente dentro del sistema?
        if t_llegada <= t and t_fin > t:
            clientes_en_sistema += 1

            # Condición 2: Si está en el sistema, ¿aún no entra al servidor? (Está en cola)
            if t_inicio > t:
                clientes_en_cola += 1

        # Actualizamos el fin anterior para el siguiente ciclo
        tiempo_fin_anterior = t_fin

    return clientes_en_sistema, clientes_en_cola

In [16]:
# SECCIÓN DE PRUEBA Y DEMOSTRACIÓN DEL ALGORITMO
# =====================================================================

# Configuración de parámetros teóricos
TASA_LLEGADAS = 2.0
TASA_SERVICIOS = 3.0
TOTAL_CLIENTES = 50

# Ejecución de la simulación
llegadas, servicios, fines = simular_tiempos_mm1(TASA_LLEGADAS, TASA_SERVICIOS, TOTAL_CLIENTES)
# Impresión de la tabla cronológica
print("-" * 90)
print("CRONOLOGÍA COMPLETA DEL SISTEMA M/M/1")
print("-" * 90)
tiempo_fin_ant = 0.0
for idx in range(TOTAL_CLIENTES):
    t_ini = max(llegadas[idx], tiempo_fin_ant)
    print(f"Cliente {idx+1:02d} | Llegada: {llegadas[idx]:6.2f} | Inicio Serv: {t_ini:6.2f} | Fin Serv: {fines[idx]:6.2f}")
    tiempo_fin_ant = fines[idx]

# Fijamos el tiempo de evaluación (por ejemplo, la llegada del 5to cliente)
tiempo_evaluacion = llegadas[4]

# Invocación del nuevo algoritmo de desglose
en_sistema, en_cola = calcular_desglose_en_tiempo_t(tiempo_evaluacion, llegadas, servicios, fines)

print("\n" + "-" * 75)
print(f"ESTADO DE LA LINEA DE ESPERA EN EL TIEMPO t = {tiempo_evaluacion:.2f}")
print("-" * 75)
print(f"-> Clientes totales en el sistema (En servicio + En cola): {en_sistema}")
print(f"-> Clientes estrictamente esperando en la cola: {en_cola}")
print(f"-> Clientes siendo atendidos en el servidor en este instante: {en_sistema - en_cola}")
print("-" * 75)

------------------------------------------------------------------------------------------
CRONOLOGÍA COMPLETA DEL SISTEMA M/M/1
------------------------------------------------------------------------------------------
Cliente 01 | Llegada:   0.13 | Inicio Serv:   0.13 | Fin Serv:   3.26
Cliente 02 | Llegada:   0.60 | Inicio Serv:   3.26 | Fin Serv:   3.26
Cliente 03 | Llegada:   0.84 | Inicio Serv:   3.26 | Fin Serv:   3.69
Cliente 04 | Llegada:   1.98 | Inicio Serv:   3.69 | Fin Serv:   3.73
Cliente 05 | Llegada:   3.01 | Inicio Serv:   3.73 | Fin Serv:   3.78
Cliente 06 | Llegada:   4.64 | Inicio Serv:   4.64 | Fin Serv:   4.76
Cliente 07 | Llegada:   4.65 | Inicio Serv:   4.76 | Fin Serv:   4.89
Cliente 08 | Llegada:   4.92 | Inicio Serv:   4.92 | Fin Serv:   5.01
Cliente 09 | Llegada:   5.81 | Inicio Serv:   5.81 | Fin Serv:   5.93
Cliente 10 | Llegada:   5.89 | Inicio Serv:   5.93 | Fin Serv:   6.04
Cliente 11 | Llegada:   5.93 | Inicio Serv:   6.04 | Fin Serv:   6.27
Cliente 12

In [20]:

# =====================================================================
# SECCIÓN: CREACIÓN Y PRESENTACIÓN FORMAL DE LA TABLA DE RESULTADOS
# =====================================================================

# 1. Calculamos los tiempos de inicio de servicio de manera precisa para el DataFrame
tiempos_inicio = []
tiempo_fin_ant = 0.0
for idx in range(TOTAL_CLIENTES):
    t_ini = max(llegadas[idx], tiempo_fin_ant)
    tiempos_inicio.append(t_ini)
    tiempo_fin_ant = fines[idx]

# 2. Estructuramos los datos en un diccionario de Python
datos_cola = {
    "ID Cliente": [f"Cliente {i+1:02d}" for i in range(TOTAL_CLIENTES)],
    "Tiempo Llegada": llegadas,
    "Tiempo Inicio Servicio": tiempos_inicio,
    "Duración Servicio": servicios,
    "Tiempo Fin Servicio": fines
}

# 3. Creamos el DataFrame de Pandas
df_resultados = pd.DataFrame(datos_cola)

# 4. Filtramos únicamente los primeros 10 clientes para la visualización formal
df_primeros_10 = df_resultados.head(10).copy()

# 5. Aplicamos un formato estético redondeado a 4 decimales para mayor limpieza visual
df_presentacion = df_primeros_10.style.format({
    "Tiempo Llegada": "{:.4f}",
    "Tiempo Inicio Servicio": "{:.4f}",
    "Duración Servicio": "{:.4f}",
    "Tiempo Fin Servicio": "{:.4f}"
}).set_properties(**{
    'text-align': 'center'  # <- Esto centra el contenido de las celdas
}).set_table_styles([
    {
        'selector': 'th',
        'props': [
            ('text-align', 'center'),
            ('border-right', '1px solid #d3d3d3'), # <- Añade la raya vertical en los títulos de las columnas
            ('border-bottom', '2px solid #808080') # <- Una raya horizontal más marcada bajo los títulos
        ]
    }
]).hide(axis="index")

# 6. Desplegamos la tabla en Google Colab
print("TABLA CRONOLÓGICA DE RENDIMIENTO DE LOS PRIMEROS 10 CLIENTES (M/M/1)\n")
display(df_presentacion)

TABLA CRONOLÓGICA DE RENDIMIENTO DE LOS PRIMEROS 10 CLIENTES (M/M/1)



ID Cliente,Tiempo Llegada,Tiempo Inicio Servicio,Duración Servicio,Tiempo Fin Servicio
Cliente 01,0.1288,0.1288,3.1335,3.2623
Cliente 02,0.5957,3.2623,0.0011,3.2635
Cliente 03,0.8401,3.2635,0.4248,3.6883
Cliente 04,1.9778,3.6883,0.0433,3.7317
Cliente 05,3.0092,3.7317,0.0501,3.7818
Cliente 06,4.6368,4.6368,0.1223,4.7590
Cliente 07,4.6524,4.7590,0.1272,4.8862
Cliente 08,4.9223,4.9223,0.0923,5.0146
Cliente 09,5.8076,5.8076,0.1175,5.9251
Cliente 10,5.8937,5.9251,0.1181,6.0432


In [21]:
def calcular_clientes_tras_llegada_n(n, tiempos_llegada, tiempos_fin_servicio):
    """
    Algoritmo que arroja el número de clientes en el sistema inmediatamente
    después de la llegada del n-ésimo cliente (donde n es un número natural >= 1).
    """
    # Validación básica de rango para el número natural n
    if n < 1 or n > len(tiempos_llegada):
        return f"Error: 'n' debe ser un número natural entre 1 y {len(tiempos_llegada)}"

    # El tiempo exacto en que llega el n-ésimo cliente (indexación 0-based de Python)
    t_n = tiempos_llegada[n - 1]

    # Contamos cuántas salidas ocurrieron estrictamente antes o en el tiempo t_n
    servicios_terminados_en_t_n = 0
    for tiempo_fin in tiempos_fin_servicio:
        if tiempo_fin <= t_n:
            servicios_terminados_en_t_n += 1

    # Inmediatamente después de esta llegada, han entrado exactamente n clientes en total
    clientes_en_sistema = n - servicios_terminados_en_t_n
    return clientes_en_sistema

In [25]:
# SECCIÓN DE PRUEBA CALCULAR CLIENTES DESPUÉS DE LLEGADA N
# =====================================================================

# Parámetros estocásticos
TASA_LLEGADAS = 2.0   # Lambda (λ)
TASA_SERVICIOS = 3.0  # Mu (μ)
TOTAL_CLIENTES = 50   # Cantidad total de clientes simulados

# 1. Ejecutar simulación global
llegadas, servicios, fines = simular_tiempos_mm1(TASA_LLEGADAS, TASA_SERVICIOS, TOTAL_CLIENTES)

# Mostrar la tabla de tiempos de control en la consola
print("-" * 90)
print("TABLA CRONOLÓGICA DE CONTROL (M/M/1)")
print("-" * 90)
for idx in range(TOTAL_CLIENTES):
    print(f"Cliente {idx+1:02d} | Llegada (t): {llegadas[idx]:6.2f} | Fin de Servicio: {fines[idx]:6.2f}")
print("-" * 90)

# 2. EVALUACIÓN DEL PUNTO 8
# Definimos el número natural 'n' que queremos inspeccionar (por ejemplo, el 6° cliente)
n_elegido = 6

# Invocar el nuevo algoritmo desarrollado
resultado_clientes = calcular_clientes_tras_llegada_n(n_elegido, llegadas, fines)

print(f"\nRESULTADO DEL PUNTO:")
print(f"-> Analizando el instante de la llegada del cliente n = {n_elegido} (Tiempo t = {llegadas[n_elegido-1]:.2f})")
print(f"-> Número de clientes en el sistema inmediatamente después de este arribo: {resultado_clientes} clientes.")
print("-" * 90)

------------------------------------------------------------------------------------------
TABLA CRONOLÓGICA DE CONTROL (M/M/1)
------------------------------------------------------------------------------------------
Cliente 01 | Llegada (t):   0.01 | Fin de Servicio:   0.97
Cliente 02 | Llegada (t):   0.23 | Fin de Servicio:   1.39
Cliente 03 | Llegada (t):   0.93 | Fin de Servicio:   1.95
Cliente 04 | Llegada (t):   1.09 | Fin de Servicio:   2.33
Cliente 05 | Llegada (t):   1.20 | Fin de Servicio:   2.43
Cliente 06 | Llegada (t):   1.26 | Fin de Servicio:   2.54
Cliente 07 | Llegada (t):   2.26 | Fin de Servicio:   2.94
Cliente 08 | Llegada (t):   2.99 | Fin de Servicio:   3.06
Cliente 09 | Llegada (t):   3.23 | Fin de Servicio:   3.37
Cliente 10 | Llegada (t):   3.48 | Fin de Servicio:   3.86
Cliente 11 | Llegada (t):   3.75 | Fin de Servicio:   4.12
Cliente 12 | Llegada (t):   5.87 | Fin de Servicio:   5.91
Cliente 13 | Llegada (t):   6.84 | Fin de Servicio:   6.92
Cliente 14 | L